In [1]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import kagglehub


In [2]:
import os
for root, dirs, files in os.walk('/kaggle'):
    for file in files:
        if 'runtime_tools' in file:
            print(os.path.join(root, file))


/kaggle/usr/lib/notebooks/mehulkumar99/runtime_tools/runtime_tools.py


In [3]:
from runtime_tools import search_value, search_column, FindShortestPath, ExecuteSQL

# Loading validation dataset

In [4]:
from datasets import load_dataset #loaded into the cpu. # dataset is a library and load_dataset is a function (as it has lowercase with underscore)

dataset = load_dataset("spider")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


In [5]:
validation_df = pd.DataFrame(dataset['validation'])
validation_df.head(10)

,db_id,query,question,query_toks,query_toks_no_value,question_toks
0,concert_singer,SELECT count(*) FROM singer,How many singers do we have?,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]","[How, many, singers, do, we, have, ?]"
1,concert_singer,SELECT count(*) FROM singer,What is the total number of singers?,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]","[What, is, the, total, number, of, singers, ?]"
2,concert_singer,"SELECT name , country , age FROM singer ORDE...","Show name, country, age for all singers ordere...","[SELECT, name, ,, country, ,, age, FROM, singe...","[select, name, ,, country, ,, age, from, singe...","[Show, name, ,, country, ,, age, for, all, sin..."
3,concert_singer,"SELECT name , country , age FROM singer ORDE...","What are the names, countries, and ages for ev...","[SELECT, name, ,, country, ,, age, FROM, singe...","[select, name, ,, country, ,, age, from, singe...","[What, are, the, names, ,, countries, ,, and, ..."
4,concert_singer,"SELECT avg(age) , min(age) , max(age) FROM s...","What is the average, minimum, and maximum age ...","[SELECT, avg, (, age, ), ,, min, (, age, ), ,,...","[select, avg, (, age, ), ,, min, (, age, ), ,,...","[What, is, the, average, ,, minimum, ,, and, m..."
5,concert_singer,"SELECT avg(age) , min(age) , max(age) FROM s...","What is the average, minimum, and maximum age ...","[SELECT, avg, (, age, ), ,, min, (, age, ), ,,...","[select, avg, (, age, ), ,, min, (, age, ), ,,...","[What, is, the, average, ,, minimum, ,, and, m..."
6,concert_singer,"SELECT song_name , song_release_year FROM sin...",Show the name and the release year of the song...,"[SELECT, song_name, ,, song_release_year, FROM...","[select, song_name, ,, song_release_year, from...","[Show, the, name, and, the, release, year, of,..."
7,concert_singer,"SELECT song_name , song_release_year FROM sin...",What are the names and release years for all t...,"[SELECT, song_name, ,, song_release_year, FROM...","[select, song_name, ,, song_release_year, from...","[What, are, the, names, and, release, years, f..."
8,concert_singer,SELECT DISTINCT country FROM singer WHERE age ...,What are all distinct countries where singers ...,"[SELECT, DISTINCT, country, FROM, singer, WHER...","[select, distinct, country, from, singer, wher...","[What, are, all, distinct, countries, where, s..."
9,concert_singer,SELECT DISTINCT country FROM singer WHERE age ...,What are the different countries with singers...,"[SELECT, DISTINCT, country, FROM, singer, WHER...","[select, distinct, country, from, singer, wher...","[What, are, the, different, countries, with, s..."


# Loading schema_lookup

In [6]:
import pickle

# Open the file in binary read mode ('rb')
with open('/kaggle/input/notebooks/mehulkumar99/preprocessing/spider_schema_lookup.pkl', 'rb') as file:
    schema_lookup = pickle.load(file)

schema_lookup['concert_singer']['embeddings'].shape

(21, 1536)

In [7]:
exemplar_1 = [
    # User message
    {
        "role": "user",
        "content": "Database Schema:\nTables: author, conference, domain, domain_author, domain_conference, journal, domain_journal, keyword, domain_keyword, publication, domain_publication, organization, publication_keyword, writes, cite\nForeignKeys:\ndomain_author.aid = author.aid\ndomain_author.did = domain.did\ndomain_conference.cid = conference.cid\ndomain_conference.did = domain.did\ndomain_journal.did = domain.did\ndomain_journal.jid = journal.jid\ndomain_keyword.did = domain.did\ndomain_keyword.kid = keyword.kid\npublication.cid = conference.cid\npublication.jid = journal.jid\ndomain_publication.did = domain.did\ndomain_publication.pid = publication.pid\npublication_keyword.pid = publication.pid\npublication_keyword.kid = keyword.kid\nwrites.aid = author.aid\nwrites.pid = publication.pid\ncite.cited = publication.pid\ncite.citing = publication.pid\n\nQuestion: Return me the author in the 'University of Michigan' in Databases area whose papers have more than 5000 total citations."
    },

    # Turn 1 — SearchColumn (skip SearchValue, DB is empty)
    {
        "role": "assistant",
        "content": "Thought: This database has no data, so SearchValue would return empty results. I will proceed directly to finding relevant columns for: author name, university name, academic domain, and total citations.",
        "tool_calls": [{
            "id": "call_001",
            "type": "function",
            "function": {
                "name": "search_column",
                "arguments": "{\"query\": [\"author name\", \"university name\", \"academic domain\", \"total citations\"]}"
            }
        }]
    },
    {
        "role": "tool",
        "content": "{'author name': [{'column': 'name', 'format': 'text', 'table': 'author', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'journal', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'conference', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'keyword', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'organization', 'statistics': 'e.g: []'}], 'university name': [{'column': 'name', 'format': 'text', 'table': 'journal', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'conference', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'author', 'statistics': 'e.g: []'}, {'column': 'homepage', 'format': 'text', 'table': 'organization', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'organization', 'statistics': 'e.g: []'}], 'academic domain': [{'column': 'did', 'format': 'number', 'table': 'domain_author', 'statistics': 'e.g: []'}, {'column': 'did', 'format': 'number', 'table': 'domain_journal', 'statistics': 'e.g: []'}, {'column': 'did', 'format': 'number', 'table': 'domain_conference', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'domain', 'statistics': 'e.g: []'}, {'column': 'did', 'format': 'number', 'table': 'domain_keyword', 'statistics': 'e.g: []'}], 'total citations': [{'column': 'cited', 'format': 'number', 'table': 'cite', 'statistics': 'e.g: []'}, {'column': 'citation_num', 'format': 'number', 'table': 'publication', 'statistics': 'e.g: []'}, {'column': 'citing', 'format': 'number', 'table': 'cite', 'statistics': 'e.g: []'}]}",
        "tool_call_id": "call_001"
    },

    # Turn 2 — SearchColumn retry
    {
        "role": "assistant",
        "content": "Thought: author.name, domain.name, and publication.citation_num are relevant. However university name was not found correctly — results show journal, conference, and author names but not a university/organization name column. A university is a type of organization, so I should search for organization name specifically.",
        "tool_calls": [{
            "id": "call_002",
            "type": "function",
            "function": {
                "name": "search_column",
                "arguments": "{\"query\": [\"organization name\"]}"
            }
        }]
    },
    {
        "role": "tool",
        "content": "[{'column': 'name', 'format': 'text', 'table': 'organization', 'statistics': 'e.g: []'}, {'column': 'homepage', 'format': 'text', 'table': 'organization', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'journal', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'conference', 'statistics': 'e.g: []'}, {'column': 'name', 'format': 'text', 'table': 'domain', 'statistics': 'e.g: []'}]",
        "tool_call_id": "call_002"
    },

    # Turn 3 — FindShortestPath
    {
        "role": "assistant",
        "content": "Thought: organization.name is the university name column. Columns to select: author.name. Columns to filter: domain.name, organization.name, publication.citation_num. I need join paths between these.",
        "tool_calls": [{
            "id": "call_003",
            "type": "function",
            "function": {
                "name": "FindShortestPath",
                "arguments": "{\"start\": [\"author.name\"], \"end\": [\"domain.name\", \"organization.name\", \"publication.citation_num\"]}"
            }
        }]
    },
    {
        "role": "tool",
        "content": "[('author.name', 'domain.name', 'author.name <-> author.aid = domain_author.aid <-> domain_author.did = domain.did <-> domain.name'), ('author.name', 'organization.name', 'author.name <-> author.oid = organization.oid <-> organization.name'), ('author.name', 'publication.citation_num', 'author.name <-> author.aid = writes.aid <-> writes.pid = publication.pid <-> publication.citation_num')]",
        "tool_call_id": "call_003"
    },

    # Turn 4 — ExecuteSQL
    {
        "role": "assistant",
        "content": "Thought: I have all join paths. The question asks for authors with more than 5000 total citations — this requires GROUP BY author.name with HAVING SUM(citation_num) > 5000.",
        "tool_calls": [{
            "id": "call_004",
            "type": "function",
            "function": {
                "name": "ExecuteSQL",
                "arguments": "{\"sql\": \"SELECT author.name FROM author JOIN domain_author ON author.aid = domain_author.aid JOIN domain ON domain_author.did = domain.did JOIN organization ON author.oid = organization.oid JOIN writes ON author.aid = writes.aid JOIN publication ON writes.pid = publication.pid WHERE domain.name = 'Databases' AND organization.name = 'University of Michigan' GROUP BY author.name HAVING SUM(publication.citation_num) > 5000\"}"
            }
        }]
    },
    {
        "role": "tool",
        "content": "[]",
        "tool_call_id": "call_004"
    },

    # Turn 5 — Done
    {
        "role": "assistant",
        "content": "Thought: The query returned empty results — this is expected as the database tables are empty. The SQL structure is correct and complete.",
        "tool_calls": [{
            "id": "call_005",
            "type": "function",
            "function": {
                "name": "Done",
                "arguments": "{}"
            }
        }]
    },
    {
    "role": "tool",
    "content": "Task Complete",
    "tool_call_id": "call_005"
}
]


exemplar_2 = [
    # User message
    {
        "role": "user",
        "content": "Database Schema:\nTables: Student, Faculty, Activity, Participates_in, Faculty_Participates_in\nForeignKeys:\nParticipates_in.StuID = Student.stuid\nParticipates_in.actid = Activity.actid\nFaculty_Participates_in.FacID = Faculty.FacID\nFaculty_Participates_in.actid = Activity.actid\n\nQuestion: Which female students and male professors participated in the Soccer activity? List their names."
    },

    # Turn 1 — SearchValue
    {
        "role": "assistant",
        "content": "Thought: The question mentions 'Soccer activity' which is likely a specific cell value. I also need to find columns for female student, male professor, and name.",
        "tool_calls": [{
            "id": "call_001",
            "type": "function",
            "function": {
                "name": "search_value",
                "arguments": "{\"query\": \"Soccer activity\", \"table\": null, \"column\": null}"
            }
        }]
    },
    {
        "role": "tool",
        "content": "[{'contents': 'Soccer', 'table': 'Activity', 'column': 'activity_name'}]",
        "tool_call_id": "call_001"
    },

    # Turn 2 — SearchColumn
    {
        "role": "assistant",
        "content": "Thought: Found 'Soccer' in Activity.activity_name. Now I need to find columns for female student, male professor, and name.",
        "tool_calls": [{
            "id": "call_002",
            "type": "function",
            "function": {
                "name": "search_column",
                "arguments": "{\"query\": [\"female student\", \"male professor\", \"name\"]}"
            }
        }]
    },
    {
        "role": "tool",
        "content": "{'female student': [{'column': 'Sex', 'format': 'text', 'table': 'Student', 'statistics': \"e.g: ['F', 'M']\"}, {'column': 'Sex', 'format': 'text', 'table': 'Faculty', 'statistics': \"e.g: ['M', 'F']\"}, {'column': 'Rank', 'format': 'text', 'table': 'Faculty', 'statistics': \"e.g: ['Instructor', 'Professor', 'AsstProf']\"}], 'male professor': [{'column': 'Sex', 'format': 'text', 'table': 'Faculty', 'statistics': \"e.g: ['M', 'F']\"}, {'column': 'Sex', 'format': 'text', 'table': 'Student', 'statistics': \"e.g: ['F', 'M']\"}, {'column': 'Rank', 'format': 'text', 'table': 'Faculty', 'statistics': \"e.g: ['Instructor', 'Professor', 'AsstProf']\"}], 'name': [{'column': 'activity_name', 'format': 'text', 'table': 'Activity', 'statistics': \"e.g: ['Soccer', 'Baseball', 'Football']\"}, {'column': 'Lname', 'format': 'text', 'table': 'Faculty', 'statistics': \"e.g: ['Kumar', 'Goldman']\"}, {'column': 'LName', 'format': 'text', 'table': 'Student', 'statistics': \"e.g: ['Smith', 'Kim']\"}]}",
        "tool_call_id": "call_002"
    },

    # Turn 3 — SearchColumn again
    {
        "role": "assistant",
        "content": "Thought: Student.Sex identifies female students. Faculty.Sex and Faculty.Rank identify male professors. However 'name' returned activity_name and only last name columns — I need both first name and last name for both Student and Faculty. Let me search more specifically.",
        "tool_calls": [{
            "id": "call_003",
            "type": "function",
            "function": {
                "name": "search_column",
                "arguments": "{\"query\": [\"first name\", \"last name\"]}"
            }
        }]
    },
    {
        "role": "tool",
        "content": "{'first name': [{'column': 'Fname', 'format': 'text', 'table': 'Faculty', 'statistics': \"e.g: ['Baruch', 'Paul', 'Brian']\"}, {'column': 'Fname', 'format': 'text', 'table': 'Student', 'statistics': \"e.g: ['Linda', 'Tracy', 'Shiela']\"}, {'column': 'activity_name', 'format': 'text', 'table': 'Activity', 'statistics': \"e.g: ['Soccer', 'Baseball']\"}], 'last name': [{'column': 'Lname', 'format': 'text', 'table': 'Faculty', 'statistics': \"e.g: ['Kumar', 'Goldman', 'Cauwenberghs']\"}, {'column': 'LName', 'format': 'text', 'table': 'Student', 'statistics': \"e.g: ['Smith', 'Kim', 'Jones']\"}]}",
        "tool_call_id": "call_003"
    },

    # Turn 4 — FindShortestPath
    {
        "role": "assistant",
        "content": "Thought: Columns to select: Student.Fname, Student.LName, Faculty.Fname, Faculty.Lname. Filter column: Activity.activity_name. The question asks for two separate groups (female students AND male professors) so I will need a UNION query. Let me find join paths for both groups.",
        "tool_calls": [{
            "id": "call_004",
            "type": "function",
            "function": {
                "name": "FindShortestPath",
                "arguments": "{\"start\": [\"Student.Fname\", \"Faculty.Fname\"], \"end\": [\"Activity.activity_name\"]}"
            }
        }]
    },
    {
        "role": "tool",
        "content": "[('Student.Fname', 'Activity.activity_name', 'Student.Fname <-> Student.stuid = Participates_in.StuID <-> Participates_in.actid = Activity.actid <-> Activity.activity_name'), ('Faculty.Fname', 'Activity.activity_name', 'Faculty.Fname <-> Faculty.FacID = Faculty_Participates_in.FacID <-> Faculty_Participates_in.actid = Activity.actid <-> Activity.activity_name')]",
        "tool_call_id": "call_004"
    },

    # Turn 5 — ExecuteSQL
    {
        "role": "assistant",
        "content": "Thought: I have both join paths. The question asks for female students AND male professors as two separate groups — this requires a UNION query, not a single JOIN.",
        "tool_calls": [{
            "id": "call_005",
            "type": "function",
            "function": {
                "name": "ExecuteSQL",
                "arguments": "{\"sql\": \"SELECT Student.Fname, Student.LName FROM Student JOIN Participates_in ON Student.stuid = Participates_in.StuID JOIN Activity ON Participates_in.actid = Activity.actid WHERE Activity.activity_name = 'Soccer' AND Student.Sex = 'F' UNION SELECT Faculty.Fname, Faculty.Lname FROM Faculty JOIN Faculty_Participates_in ON Faculty.FacID = Faculty_Participates_in.FacID JOIN Activity ON Faculty_Participates_in.actid = Activity.actid WHERE Activity.activity_name = 'Soccer' AND Faculty.Sex = 'M' AND Faculty.Rank = 'Professor'\"}"
            }
        }]
    },
    {
        "role": "tool",
        "content": "[('Linda', 'Smith'), ('Sarah', 'Schmidt'), ('Stacy', 'Prater'), ('Michael', 'Goodrich')]",
        "tool_call_id": "call_005"
    },

    # Turn 6 — Done
    {
        "role": "assistant",
        "content": "Thought: The UNION query executed successfully returning female students and male professors who participated in Soccer.",
        "tool_calls": [{
            "id": "call_006",
            "type": "function",
            "function": {
                "name": "Done",
                "arguments": "{}"
            }
        }]
    },
    {
    "role": "tool",
    "content": "Task Complete",
    "tool_call_id": "call_006"
}
]

# TOOL definitions

In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_column",
            "description": "Search for relevant columns in the database based on semantic meaning. Use this to find which columns are related to concepts in the question. Returns up to 5 most relevant columns per query term, and corresponding tables, data types, and sample statistics.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "List of semantic descriptions of columns you are looking for. Even for a single search term, wrap it in a list. e.g. ['singer name'] or ['singer name', 'song release year']"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            },
            "strict": True
        }
    },

    {
    "type": "function",
    "function": {
        "name": "search_value",
        "description": "This tool searches for a value in the database based on the query parameter. It utilizes SQLite LIKE query to find the most relevant cell value, and returns the actual value, the corresponding column name, and the table name. You can also specify the table and column parameters to search within a specific table or column. If these parameters are not specified, the tool will search across all tables and columns. Note: This tool does not index the id column because the id column is not a text field.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The text snippet or value to look up in the database. Examples: 'John Doe', 'California'."
                },
                "table": {
                    "type": [
                        "string",
                        "null"
                    ],
                    "description": "The name of the specific table to restrict the search to. If you want to search all tables, you must explicitly pass null."
                },
                "column": {
                    "type": [
                        "string",
                        "null"
                    ],
                    "description": "The name of the specific column to restrict the search to. If you want to search all columns, you must explicitly pass null."
                }
            },
            "required": [
                "query",
                "table",
                "column"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
},
    
    {
        "type": "function",
        "function": {
            "name": "FindShortestPath",
            "description": "This function identifies the shortest path between two sets of columns in a database schema, treated as an undirected linked graph. In this graph, nodes represent columns, one type of edges represent foreign key constraints (linking tables) and another type of edge represent all the columns in a table joined to each other within the table (for in-table traversal) . The parameters start and end accept either a single column or a list of columns specified in the {table}.{column} format. The function returns the shortest path between each possible pair of start and end nodes, which can be utilized to construct SQL queries. In the returned path, <-> indicates traversal within the same table, and = indicates a foreign key join condition between two tables. Use the = conditions directly as JOIN ON clauses in your SQL query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "List of starting column nodes specified in the {table}.{column} format. Even for a single column, wrap it in a list. Examples: ['users.id'] or ['users.id', 'orders.user_id']"
                    },
                    "end": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "List of destination column nodes specified in the {table}.{column} format. Even for a single column, wrap it in a list. Examples: ['products.id'] or ['products.id', 'categories.id']"
                    }
                },
                "required": [
                    "start",
                    "end"
                ],
                "additionalProperties": False
            },
            "strict": True
        }
    },

        {
        "type": "function",
        "function": {
            "name": "ExecuteSQL",
            "description": "Executes a SQL query against the database. Returns the result rows if successful, or an error message if the query fails. The result of the final ExecuteSQL call will be taken as the answer — ensure the query is complete and accurate before calling this.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "The complete, valid SQL query to execute against the database. Example: 'SELECT name FROM users WHERE id = 1;'"
                    }
                },
                "required": [
                    "sql"
                ],
                "additionalProperties": False
            },
            "strict": True
        }
    },

    {
    "type": "function",
    "function": {
        "name": "Done",
        "description": "Call this when you have executed the final SQL query using ExecuteSQL and have the answer. Only call this after ExecuteSQL has returned the result rows.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        },
        "strict": True
    }
}
    

]


## system instructions

In [9]:
system_prompt = """When presented with a question, you need to use specific tools to interact with a locally deployed SQLite database and craft an SQL query to retrieve the answer.

General Process: 

1. If a question includes a specific string value or entity name, utilize the SearchValue tool to identify the corresponding cell value, column name, and table name. 
2. Use SearchColumn to locate the column that is semantically related to the query. 
3. Determine the shortest path between the selected columns and the condition columns using FindShortestPath 
4. Execute the SQL query using ExecuteSQL. The result from the final call to ExecuteSQL will be taken as the answer, so it is critical to ensure that the SQL query is accurately formulated and complete. 

Crucial Patterns:
- You need to use your intelligence and reasoning to determine how columns in the database reflect the semantics of the question, then use SearchColumn() to locate them. For instance, "the city Bishopville, SC" may be represented by two columns: T.city = 'Bishopville' AND T.state = 'SC'.
- Be aware of dirty data in the database. Use 'IS NOT NULL' to filter out missing values.- MUST construct SQL queries strictly according to the observed values. For example, 'Akiak ' includes a trailing space.
- DO NOT select irrelevant columns. (e.g. for the question "who has the max value", you should NOT return the max value itself, but if the question is "what is the max value, list the people", you should return the max value and the corresponding people.)
- Use numbers from 0 to 100 to represent percentages. For example, use *100 in SQL if necessary.
- DO NOT concatenate column in select statement. For example, `SELECT c1 || ' ' || c2` is not allowed. You should SELECT c1, c2 instead. 

Remember:
- DO NOT fabricate column names. Column names must be based on actual observation.
- Call only ONE tool at a time per turn.
- Always generate a Thought before calling a tool.
- STOP generating after calling a tool — wait for the observation.
- Only call Done after ExecuteSQL has returned the final result rows.          
- Please keep interactions concise and accurate, avoiding unnecessary details. 
"""

In [10]:
validation_df.head(3)

,db_id,query,question,query_toks,query_toks_no_value,question_toks
0,concert_singer,SELECT count(*) FROM singer,How many singers do we have?,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]","[How, many, singers, do, we, have, ?]"
1,concert_singer,SELECT count(*) FROM singer,What is the total number of singers?,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]","[What, is, the, total, number, of, singers, ?]"
2,concert_singer,"SELECT name , country , age FROM singer ORDE...","Show name, country, age for all singers ordere...","[SELECT, name, ,, country, ,, age, FROM, singe...","[select, name, ,, country, ,, age, from, singe...","[Show, name, ,, country, ,, age, for, all, sin..."


In [11]:
def build_initial_message(question, db_id, schema_lookup):

    schema_str = schema_lookup[db_id]['schema']
    messages = []
    messages.append({'role':'system',
                   'content': system_prompt})
    messages.extend(exemplar_2)
    messages.extend(exemplar_1)

    messages.append({'role':'user',
                   'content': f'\nDatabase_Schema: {schema_str}\n\nQuestion: {question}'})

    return messages

In [12]:
question = 'How many singers are there'
db_id = 'concert_singer'
messages = build_initial_message(question, db_id, schema_lookup)
print(messages)

[{'role': 'system', 'content': 'When presented with a question, you need to use specific tools to interact with a locally deployed SQLite database and craft an SQL query to retrieve the answer.\n\nGeneral Process: \n\n1. If a question includes a specific string value or entity name, utilize the SearchValue tool to identify the corresponding cell value, column name, and table name. \n2. Use SearchColumn to locate the column that is semantically related to the query. \n3. Determine the shortest path between the selected columns and the condition columns using FindShortestPath \n4. Execute the SQL query using ExecuteSQL. The result from the final call to ExecuteSQL will be taken as the answer, so it is critical to ensure that the SQL query is accurately formulated and complete. \n\nCrucial Patterns:\n- You need to use your intelligence and reasoning to determine how columns in the database reflect the semantics of the question, then use SearchColumn() to locate them. For instance, "the ci

# loop for one example

```initialize messages
while turn < max_turns:
    call API
    read response
    if tool_name == "Done": break
    extract tool_name + arguments
    call corresponding Python function
    append assistant message + tool observation to messages
    turn += 1
store last_sql + full conversation

In [13]:
import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Openai") 

client = OpenAI(api_key=api_key)
MAX_TURNS = 12

In [14]:
def run_agent(question, db_id, schema_lookup, db_base_path):

    """
    Runs the full ReAct loop for one question.
    Returns dict with predicted_sql, conversation, turns_used, hit_max_turns.
    """

    turn = 0  
    last_sql = None 
    status = 'SQL never generated'
    messages = build_initial_message(question, db_id, schema_lookup)
    turn_2_sql_attempts_map = {}

    while turn < MAX_TURNS:

        # api call # takes in message, tools, 
        response = client.chat.completions.create(
            model = 'gpt-4o-mini',
            messages = messages, 
            tools = tools,
            temperature = 0.0
        ) 

        message = response.choices[0].message  #  OpenAI SDK object (a Pydantic-style model), not a raw dict, you access its fields as attributes, not via ['key'] indexing.
        messages.append(message)
        
        tool_calls = message.tool_calls

        if not tool_calls:
            break

        tool_call = tool_calls[0]
        tool_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        tool_id = tool_call.id

        if tool_name == 'search_value':

            results =  search_value(query = args['query'],
                                    db_id = db_id, 
                                    col_dicts = schema_lookup[db_id]['col_dicts'] , 
                                    db_base_path = db_base_path,
                                    table = args['table'],
                                    column = args['column'])
        elif tool_name == 'search_column':

            results = search_column(query = args['query'],
                                   col_dicts = schema_lookup[db_id]['col_dicts'],
                                   embeddings = schema_lookup[db_id]['embeddings'],
                                   model = model,
                                   top_k = 5)
        

        elif tool_name == 'FindShortestPath':
            results = FindShortestPath(db_id = db_id,
                                       schema_lookup = schema_lookup,
                                       start = args['start'],
                                       end = args['end'])
            
        elif tool_name == 'ExecuteSQL':
            sql = args['sql']
            last_sql = sql
            status, results = ExecuteSQL(sql = sql,
                                 db_id = db_id,
                                 db_base_path = db_base_path)

            if status =='Successful' and isinstance(results, list) and len(results)>10:
                
                display_result = results[:10]
                observation = f'{display_result}....{len(results)} total rows, showing first 10'
                
            else:
                observation = results

            turn_2_sql_attempts_map[turn] = {'sql': sql, 'status': status, 'result': observation}
            
            # sql_attempts.append({'turn': turn, 'sql': sql, 'status': status, 'result': observation})

            
        elif tool_name == 'Done':
            messages.append({
                'role': "tool",
                'content': 'Task Complete', 
                'tool_call_id': tool_id
            })
            turn+=1
            break

        else:
            results = f'Unknown tool name: {tool_name}'

        # Append tool observation to history
        messages.append({
            "role": "tool",
            "content": str(results),
            "tool_call_id": tool_id
        })
        
        turn += 1

    return {
        "db_id": db_id,
        "question": question,
        "conversation": messages,
        "turns_used": turn,
        "hit_max_turns": turn == MAX_TURNS,
        "predicted_sql": last_sql,
        "sql_status": status,
        "sql_attempts": turn_2_sql_attempts_map,

    }

    

In [15]:
index = 0
question = validation_df['question'].iloc[index]
db_id = validation_df['db_id'].iloc[index]
db_base_path = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'

result = run_agent(question, db_id, schema_lookup, db_base_path)
sql = result['predicted_sql']
conversation = result['conversation']
print(sql)
print(conversation)

SELECT COUNT(*) AS total_singers FROM singer;
[{'role': 'system', 'content': 'When presented with a question, you need to use specific tools to interact with a locally deployed SQLite database and craft an SQL query to retrieve the answer.\n\nGeneral Process: \n\n1. If a question includes a specific string value or entity name, utilize the SearchValue tool to identify the corresponding cell value, column name, and table name. \n2. Use SearchColumn to locate the column that is semantically related to the query. \n3. Determine the shortest path between the selected columns and the condition columns using FindShortestPath \n4. Execute the SQL query using ExecuteSQL. The result from the final call to ExecuteSQL will be taken as the answer, so it is critical to ensure that the SQL query is accurately formulated and complete. \n\nCrucial Patterns:\n- You need to use your intelligence and reasoning to determine how columns in the database reflect the semantics of the question, then use Search

In [16]:
print(len(conversation))

30


In [17]:
conversation[26]

ChatCompletionMessage(content='Thought: I need to count the number of unique singers in the singer table. I will construct a SQL query to do that.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ruinlIIvu5moo8T2BKaDiE4W', function=Function(arguments='{"sql":"SELECT COUNT(*) AS total_singers FROM singer;"}', name='ExecuteSQL'), type='function')])

In [18]:
print(conversation[26].role)
print('-----------------------')
print(conversation[26].content)
print('-----------------------')
print(conversation[26].tool_calls)
print('-----------------------')
print(conversation[26].tool_calls[0].function.name)
print('-----------------------')
print(conversation[26].tool_calls[0].function.arguments)
print('-----------------------')
print(json.loads(conversation[26].tool_calls[0].function.arguments)['sql'])

assistant
-----------------------
Thought: I need to count the number of unique singers in the singer table. I will construct a SQL query to do that.
-----------------------
[ChatCompletionMessageFunctionToolCall(id='call_ruinlIIvu5moo8T2BKaDiE4W', function=Function(arguments='{"sql":"SELECT COUNT(*) AS total_singers FROM singer;"}', name='ExecuteSQL'), type='function')]
-----------------------
ExecuteSQL
-----------------------
{"sql":"SELECT COUNT(*) AS total_singers FROM singer;"}
-----------------------
SELECT COUNT(*) AS total_singers FROM singer;


In [19]:
turn = 0


f_string = f"""============================================================
FINAL RESULT
============================================================
"""

In [20]:
def print_conversation(conversation, result):

    turn = 0
    turn_2_sql_attempts_map = result['sql_attempts']
    first_call_initiated = False

    for msg in conversation:
        if not isinstance(msg, dict) and msg.role =='assistant':

            misc = f"""============================================================
TURN {turn}
============================================================\n"""

            first_call_initiated = True

            print(misc)
            print('[ASSISTANT - Thought]')
            print(f'{msg.content}\n')

            tool_calls = msg.tool_calls
            tool_call = tool_calls[0]
            tool_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            temp =''
            for k,v in args.items():
                temp+= f'{k} = {v}, '
                

            print('[ASSISTANT - Action]')
            print(f'-> {tool_name}( {temp} )\n')

        elif isinstance(msg, dict) and msg['role'] =='tool' and first_call_initiated:

            print('[TOOL - Observation]')

            if tool_name == 'ExecuteSQL':

                dic = turn_2_sql_attempts_map[turn]
                
                print(f"Status: {dic['status']}")
                print(f"Result: {msg['content']}\n")
            else:
                print(f"{msg['content']}\n")

            turn+=1

    print(f'{f_string}')
    print(f"Predicted_sql: {result['predicted_sql']}")
    print(f"SQL status: {result['sql_status']}")
    print(f"Turns used: {result['turns_used']}")
    print(f"Hit Max Turns : {result['turns_used']== MAX_TURNS}")
    # print(f'Result : {result}' )

In [21]:
output = print_conversation(conversation, result)
output

TURN 0

[ASSISTANT - Thought]
Thought: I need to count the number of unique singers in the singer table. I will construct a SQL query to do that.

[ASSISTANT - Action]
-> ExecuteSQL( sql = SELECT COUNT(*) AS total_singers FROM singer;,  )

[TOOL - Observation]
Status: Successful
Result: [(6,)]

TURN 1

[ASSISTANT - Thought]
Thought: The query executed successfully, returning a count of 6 singers. 

Now I will finalize the task.

[ASSISTANT - Action]
-> Done(  )

[TOOL - Observation]
Task Complete

FINAL RESULT

Predicted_sql: SELECT COUNT(*) AS total_singers FROM singer;
SQL status: Successful
Turns used: 2
Hit Max Turns : False


In [22]:
import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Openai") 

client = OpenAI(api_key=api_key)

MAX_TURNS = 12

def run_agent(question, db_id, schema_lookup, db_base_path):
    """
    Runs the full ReAct loop for one question.
    Returns dict with predicted_sql, conversation, turns_used, hit_max_turns.
    """
    
    # Initialize messages
    messages = build_initial_messages(question, db_id, schema_lookup)
    
    last_sql = None
    turn = 0
    
    while turn < MAX_TURNS:
        
        # API call
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            temperature=0.0
        )
        
        message = response.choices[0].message
        
        # Append assistant message to history
        messages.append(message)
        
        # Check if model called a tool
        if not message.tool_calls:
            # No tool called — unexpected, treat as done
            break
        
        tool_call = message.tool_calls[0]
        tool_name = tool_call.function.name
        tool_id = tool_call.id
        arguments = json.loads(tool_call.function.arguments)
        
        # Check for Done
        if tool_name == "Done":
            break
        
        # Execute the appropriate tool
        if tool_name == "search_column":
            result = search_column(
                query=arguments["query"],
                col_dicts=schema_lookup[db_id]["col_dicts"],
                embeddings=schema_lookup[db_id]["embeddings"],
                model=embedding_model
            )
            
        elif tool_name == "search_value":
            result = search_value(
                query=arguments["query"],
                db_id=db_id,
                col_dicts=schema_lookup[db_id]["col_dicts"],
                db_base_path=db_base_path,
                table=arguments.get("table"),
                column=arguments.get("column")
            )
            
        elif tool_name == "FindShortestPath":
            result = FindShortestPath(
                db_id=db_id,
                schema_lookup=schema_lookup,
                start=arguments["start"],
                end=arguments["end"]
            )
            
        elif tool_name == "ExecuteSQL":
            sql = arguments["sql"]
            last_sql = sql  # track last SQL
            result = ExecuteSQL(
                sql=sql,
                db_id=db_id,
                db_base_path=db_base_path
            )
        
        else:
            result = f"Unknown tool: {tool_name}"
        
        # Append tool observation to history
        messages.append({
            "role": "tool",
            "content": str(result),
            "tool_call_id": tool_id
        })
        
        turn += 1
    
    return {
        "db_id": db_id,
        "question": question,
        "predicted_sql": last_sql,
        "conversation": messages,
        "turns_used": turn,
        "hit_max_turns": turn == MAX_TURNS
    }


def run_full_eval(dataset, schema_lookup, db_base_path):
    """
    Runs the agent on all examples in the dataset.
    Saves results to JSON after each example.
    """
    results = []
    
    for i, example in enumerate(dataset):
        question = example["question"]
        db_id = example["db_id"]
        gold_sql = example["query"]
        
        print(f"[{i+1}/1034] DB: {db_id} | Q: {question[:50]}...")
        
        try:
            result = run_agent(question, db_id, schema_lookup, db_base_path)
            result["gold_sql"] = gold_sql
            results.append(result)
            
        except Exception as e:
            print(f"  ERROR: {e}")
            results.append({
                "db_id": db_id,
                "question": question,
                "gold_sql": gold_sql,
                "predicted_sql": None,
                "error": str(e),
                "turns_used": 0,
                "hit_max_turns": False
            })
        
        # Save after every example — don't lose progress if it crashes
        with open("results.json", "w") as f:
            json.dump(results, f, indent=2)
    
    print(f"\nDone. {len(results)} examples processed.")
    return results